In [ ]:
# Install necessary libraries
!pip install pandas spacy sentence-transformers langchain langchain-groq streamlit
# Import necessary libraries
import pandas as pd
import spacy
import json
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer, util
import streamlit as st
import PyPDF2
import docx
import io
import pickle
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 43.5 MB/s eta 0:00:00


In [2]:
# Download the spaCy 'en_core_web_sm' model
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
resume_df = pd.read_csv('/content/Resume.csv')
resume_data_df = pd.read_csv('/content/resume_data.csv')

# Display the first 2 rows of resume_df to verify loading
print("First 2 rows of 'Resume.csv':")
display(resume_df.head(2))

# Display the first 2 rows of resume_data_df to verify loading
print("\nFirst 2 rows of 'resume_data.csv':")
display(resume_data_df.head(2))

First 2 rows of 'Resume.csv':


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR



First 2 rows of 'resume_data.csv':


,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,﻿job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.85
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.75


In [ ]:
nlp = spacy.load('en_core_web_sm')

def redact_resume(text):
    """
    Redacts Personally Identifiable Information (PII) from the given text
    using spaCy's en_core_web_sm model.
    Specifically targets 'PERSON', 'DATE', and 'ORG' entities.
    """
    doc = nlp(text)
    redacted_text = list(text) # Convert to list to allow mutable modifications

    for ent in sorted(doc.ents, key=lambda x: x.start_char, reverse=True):
        if ent.label_ == 'PERSON':
            placeholder = '[PERSON]'
        elif ent.label_ == 'DATE':
            placeholder = '[DATE]'
        elif ent.label_ == 'ORG':
            placeholder = '[ORG]'
        else:
            continue 

        
        redacted_text[ent.start_char:ent.end_char] = list(placeholder)

    return "".join(redacted_text)

# Test case with a dummy sentence
dummy_sentence = "John Doe, a software engineer, worked at Google from 2018 to 2023. He was born on January 1, 1990."
redacted_sentence = redact_resume(dummy_sentence)

print("Original Sentence:", dummy_sentence)
print("Redacted Sentence:", redacted_sentence)

Original Sentence: John Doe, a software engineer, worked at Google from 2018 to 2023. He was born on January 1, 1990.
Redacted Sentence: [PERSON], a software engineer, worked at [ORG] from [DATE] to 2023. He was born on [DATE].


In [ ]:
def extract_resume_data(redacted_text, groq_api_key):
    """
    Extracts structured data from redacted resume text using LangChain and Groq API.

    Args:
        redacted_text (str): The resume text with PII redacted.
        groq_api_key (str): Your Groq API key.

    Returns:
        dict: A dictionary containing extracted 'skills', 'education_level', and 'years_of_experience',
              or an error message if extraction fails or JSON is invalid.
    """
    # Initialize the ChatGroq model with the specified model name
    llm = ChatGroq(temperature=0, groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant") # Updated model name

    # Define the prompt template for data extraction
    # It's crucial to explicitly ask for strict JSON format
    prompt_template = PromptTemplate.from_template(
        """You are an expert resume parser. Extract the following information from the provided resume text into a strict JSON object. If a piece of information is not found, use 'N/A' for strings or an appropriate default (e.g., 0 for years of experience if not found).

        Resume Text: {text}

        Expected JSON format:
        {{
            "skills": ["skill1", "skill2", ...],
            "education_level": "highest degree obtained",
            "years_of_experience": integer
        }}

        Return ONLY the JSON object and nothing else. Ensure the output is valid and strict JSON.
        """
    )

    # Create a chain to process the prompt and get a response from the LLM
    chain = prompt_template | llm

    # Invoke the chain with the redacted text
    response = chain.invoke({"text": redacted_text})

    # Attempt to parse the response as JSON with error handling
    try:
        extracted_data = json.loads(response.content)
        # Validate the structure of the extracted data
        if all(key in extracted_data for key in ["skills", "education_level", "years_of_experience"]):
            # Ensure years_of_experience is an integer
            if isinstance(extracted_data.get("years_of_experience"), (int, float)):
                extracted_data["years_of_experience"] = int(extracted_data["years_of_experience"])
            elif extracted_data.get("years_of_experience") == 'N/A':
                extracted_data["years_of_experience"] = 0 

            return extracted_data
        else:
            return {"error": "LLM returned incomplete JSON structure.", "llm_output": response.content}
    except json.JSONDecodeError as e:
        return {"error": f"LLM did not return valid JSON: {e}", "llm_output": response.content}
    except Exception as e:
        return {"error": f"An unexpected error occurred: {e}", "llm_output": response.content}


In [ ]:
# --- Test Case for extract_resume_data ---
# The Groq API key provided by the user.
GROQ_API_KEY = 'Pate your key'

# Example redacted resume text (using the previously redacted sentence, expanded for more context)
example_redacted_resume = "[PERSON] is a software engineer with 5 years of experience, specializing in Python, Java, and Machine Learning. [PERSON] holds a Master's degree in Computer Science from [ORG] and a Bachelor's from another [ORG]. [PERSON] worked at [ORG] from [DATE] to [DATE], contributing to various projects. [PERSON] also has experience in project management. Education: Master of Science in Computer Science."

extracted_info = extract_resume_data(example_redacted_resume, GROQ_API_KEY)

print("\nExtracted Information:")
print(json.dumps(extracted_info, indent=4))


Extracted Information:
{
    "skills": [
        "Python",
        "Java",
        "Machine Learning"
    ],
    "education_level": "Master's",
    "years_of_experience": 5
}


In [ ]:
# Load the Sentence Transformer model once
# This model will be used for generating embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_match_score(candidate_skills_list, job_description_text):
    """
    Calculates a match score between a candidate's skills and a job description
    using sentence embeddings and cosine similarity.

    Args:
        candidate_skills_list (list): A list of strings representing the candidate's skills.
        job_description_text (str): The text of the job description.

    Returns:
        float: The match score as a percentage, rounded to 2 decimal places.
    """
    # Join candidate skills into a single string for embedding
    candidate_skills_str = ", ".join(candidate_skills_list)

    # Generate embeddings for both the candidate's skills and the job description
    candidate_embedding = model.encode(candidate_skills_str, convert_to_tensor=True)
    job_description_embedding = model.encode(job_description_text, convert_to_tensor=True)

    # Calculate cosine similarity
    cosine_similarity = util.cos_sim(candidate_embedding, job_description_embedding)

    # Convert similarity to a percentage and round to 2 decimal places
    match_score_percentage = round(cosine_similarity.item() * 100, 2)

    return match_score_percentage

# --- Test Case ---
candidate_skills = ['Python', 'Pandas', 'Scikit-learn']
job_description = 'Data Scientist with Machine Learning experience'

score = calculate_match_score(candidate_skills, job_description)
print(f"\nMatch Score: {score}%")

candidate_skills_2 = ['Java', 'Spring Boot', 'Microservices']
job_description_2 = 'Senior Software Engineer with extensive experience in backend development using Java and Spring Boot.'
score_2 = calculate_match_score(candidate_skills_2, job_description_2)
print(f"Match Score for another candidate: {score_2}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Match Score: 21.95%
Match Score for another candidate: 44.91%


In [ ]:
def generate_feedback(candidate_skills_list, job_description_text, groq_api_key):
    """
    Generates feedback on missing skills for a candidate based on a job description
    using a ChatGroq LLM acting as an HR advisor.

    Args:
        candidate_skills_list (list): A list of strings representing the candidate's skills.
        job_description_text (str): The text of the job description.
        groq_api_key (str): Your Groq API key.

    Returns:
        str: A 3-sentence summary of core missing skills from the LLM.
    """
    llm = ChatGroq(temperature=0.3, groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant")

    prompt_template = PromptTemplate.from_template(
        """You are an experienced HR advisor. A candidate with the following skills: {candidate_skills} is applying for a role with the following job description: {job_description}.
        Please provide a short, 3-sentence summary of what core skills the candidate appears to be missing based on the job description. Focus only on the skills.
        """
    )

    chain = prompt_template | llm
    response = chain.invoke({
        "candidate_skills": ", ".join(candidate_skills_list),
        "job_description": job_description_text
    })

    return response.content

# --- Test Case ---
candidate_skills_for_feedback = ['Python', 'SQL', 'Data Visualization']
job_description_for_feedback = 'We are looking for a Senior Data Scientist with expertise in Machine Learning, Deep Learning, Python, and cloud platforms like AWS. Experience with model deployment is a plus.'

feedback = generate_feedback(
candidate_skills_for_feedback,
job_description_for_feedback,
GROQ_API_KEY # Reusing the previously defined GROQ_API_KEY
)

print("\nCandidate Feedback:")
print(feedback)


Candidate Feedback:
The candidate appears to be missing expertise in Machine Learning, Deep Learning, and experience with model deployment. Additionally, they lack experience with cloud platforms like AWS.
